To open on Google Colab\
https://colab.research.google.com/github/RodrigoAVargasHdz/CHEM-4PB3/blob/main/Course_Notes/Week3/linear_kinetic_models.ipynb

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold

import matplotlib 
import matplotlib.pyplot as plt

## Regression model to predict ```logP``` (*lipophilicity*)

* ```logP``` is one of the most used observalbes in drug design. 
* It measures how much of a solute dissolves in the water portion versus an organic portion.
* $\log P < 0$, the compound has a higher affinity for the aqueous phase.
* $\log P > 0$, higher concentration in the lipid phase.
* $\log P = 0$, the compound is equally partitioned between the lipid and aqueous phases.

**Drug Discovery**\
```logP``` is used in the pharmaceutical/biotech industries to understand the behavior of drug molecules in the body. Drug candidates are often screened according to ```logP```, among other criteria, to help guide drug selection and analog optimization. This is because lipophilicity is a major determining factor in a
compound?s absorption, distribution in the body, penetration across vital membranes and biological barriers, metabolism and excretion. 


**Reference**:\
[**link**](https://www.acdlabs.com/wp-content/uploads/download/app/physchem/making_sense.pdf)

## Example
[**paper**](https://doi.org/10.1016/S0960-894X(03)00656-5)\
[**example**](https://kbbox.h-its.org/toolbox/tutorials/data-exploration-and-linear-regression-of-a-kinetic-dataset-using-r/)\
[**cross-validation**](https://scikit-learn.org/stable/modules/cross_validation.html)

data set,
```python
data_url = 'https://kbbox.h-its.org/media/uploads/2017/11/15/map38.csv' 
```

In [ ]:
#load data
data_url = 'https://kbbox.h-its.org/media/uploads/2017/11/15/map38.csv'
data = pd.read_csv(data_url)

# * Print the name of the columns in the file
columns_names = data.columns
print(columns_names)

In [ ]:
y = data['lipophilicity'].to_numpy()
X = data.drop(['compound_key', 'lipophilicity'],axis=1).to_numpy()
print('Data dimensions: ',X.shape)

# split training and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)


### Models

In [ ]:
def linear_model_solver(data, deg, l):
    X, y = data
    poly = PolynomialFeatures(deg)
    # X = x[np.newaxis].T  # resize to (N,d)
    Phi = poly.fit_transform(X)    
    Phit_Phi = Phi.T @ Phi
    Phit_y = Phi.T @ y

    lambda_I = l*np.eye(Phit_Phi.shape[0])  # regularization

    A = Phit_Phi + lambda_I
    A_inv = np.linalg.inv(A)
    w = A_inv @ Phit_y
    return w


In [ ]:
def f_poly(X, w, deg):
    poly = PolynomialFeatures(deg)
        
    Phi = poly.fit_transform(X)
    y_pred = Phi @ w
    return y_pred

In [ ]:
def f_rmse(y_true, y_pred):
    z = (y_true - y_pred)**2
    return np.sqrt(np.mean(z))


In [ ]:
data = (X_train,y_train)
data_val = (X_val,y_val)
deg = 2
# l_ = np.array([0.,1E-4,1E-3,1E-2,1E-1,1.])
l_ = np.exp(np.linspace(-6.,0,10))
l_ = np.sort(np.append(l_,0.))

rmse_ = []
for l in l_:
    w = linear_model_solver(data,deg,l)
    y_val_pred = f_poly(X_val,w,deg)
    rmse = f_rmse(y_val,y_val_pred)
    print('l =  %.6f, rmse = %.3f '%(l,rmse))
    rmse_.append(rmse)


In [ ]:
plt.figure(0)
plt.plot(np.log(l_),rmse_,ms=8,marker='o')

lambda_ticks = [r'$0$', r'$10^{-4}$', r'$10^{-3}$',
                r'$10^{-2}$', r'$10^{-1}$', r'$10^{1}$']
# plt.xticks(np.arange(len(l_)),lambda_ticks)
plt.ylabel('RMSE (validation)',fontsize=15)
plt.xlabel(r'$\log(\lambda)$',fontsize=15)
# plt.xlim(-0.1,1.1)

## K-fold cross validation
![k-fold]([Figures/](https://raw.github.com/RodrigoAVargasHdz/CHEM-4PB3/master/Course_Notes/Figures/grid_search_cross_validation.png))

In [ ]:
kf = KFold(n_splits=5)
for train, val in kf.split(X):
    X_train, y_train = X[train],y[train]
    X_val, y_val = X[val],y[val]
    